# LLM, tokens y límites de contexto

Un LLM predice el siguiente token. Los tokens definen coste y tamaño de contexto. Texto fluido no es lo mismo que verdad.


## Tokens y coste

Las APIs cobran y truncan en **tokens**, no en caracteres. Cuenta entrada y salida por separado.


In [ ]:
def estimate_tokens(text: str) -> int:
    """Prefer tiktoken when available; otherwise ≈4 characters per token."""
    try:
        import tiktoken  # type: ignore

        enc = tiktoken.get_encoding("cl100k_base")
        return len(enc.encode(text))
    except Exception:
        return max(1, (len(text) + 3) // 4)


def estimate_cost_usd(
    prompt_tokens: int,
    completion_tokens: int,
    *,
    in_per_m: float = 0.15,
    out_per_m: float = 0.60,
) -> float:
    return (prompt_tokens * in_per_m + completion_tokens * out_per_m) / 1_000_000


prompt = "Explain tokens in one sentence."
completion = "Tokens are the pieces of text a model reads and writes."
pt, ct = estimate_tokens(prompt), estimate_tokens(completion)
print(f"prompt tokens ≈ {pt}")
print(f"completion tokens ≈ {ct}")
print(f"estimated cost ≈ ${estimate_cost_usd(pt, ct):.6f}")

samples = [
    "Hello",
    "Olá, tudo bem?",
    "def add(a, b):\n    return a + b\n",
]
for s in samples:
    print(repr(s), "→", estimate_tokens(s), "tokens")


## Ventana de contexto finita

System prompt, historial, docs recuperados y espacio para la respuesta comparten el mismo presupuesto. El desborde causa truncado o error.


In [ ]:
from dataclasses import dataclass

@dataclass
class ContextBudget:
    window: int = 8_000
    system_tokens: int = 0
    history_tokens: int = 0
    retrieval_tokens: int = 0
    reserve_output: int = 500

    def used(self) -> int:
        return self.system_tokens + self.history_tokens + self.retrieval_tokens

    def remaining_for_input(self) -> int:
        return self.window - self.used() - self.reserve_output

    def fit_chunks(self, chunks: list[str]) -> list[str]:
        kept: list[str] = []
        room = self.remaining_for_input()
        for chunk in chunks:
            need = estimate_tokens(chunk)
            if need <= room:
                kept.append(chunk)
                room -= need
            else:
                break
        return kept

budget = ContextBudget(system_tokens=200, history_tokens=1_500, reserve_output=500)
chunks = [
    "Doc A: refund policy " + ("policy " * 40),
    "Doc B: shipping times " + ("ship " * 40),
    "Doc C: unused legal appendix " + ("legal " * 200),
]
kept = budget.fit_chunks(chunks)
print("window:", budget.window)
print("used before retrieval:", budget.used())
print("room for retrieval:", budget.remaining_for_input())
print("chunks kept:", len(kept), "/", len(chunks))
for i, c in enumerate(kept):
    print(f"  kept[{i}] tokens ≈", estimate_tokens(c))


## Probabilidad ≠ verdad

Mayor temperatura aplana la distribución: tokens más sorprendentes, también erróneos.


In [ ]:
import random

def sample_next(candidates: list[tuple[str, float]], temperature: float = 1.0) -> str:
    words, weights = zip(*candidates)
    if temperature <= 0:
        return words[weights.index(max(weights))]
    adjusted = [w ** (1.0 / temperature) for w in weights]
    total = sum(adjusted)
    probs = [a / total for a in adjusted]
    return random.choices(words, weights=probs, k=1)[0]

# Toy next-token distribution for the prefix "The capital of Brazil is"
candidates = [("Brasília", 0.70), ("São Paulo", 0.15), ("Rio", 0.10), ("Atlantis", 0.05)]
random.seed(0)
print("temperature=0.2 →", [sample_next(candidates, 0.2) for _ in range(8)])
print("temperature=1.5 →", [sample_next(candidates, 1.5) for _ in range(8)])
print("Fluent is not the same as true: low-probability tokens can still be sampled.")


## Conclusión

1. Los tokens impulsan coste, latencia y lo que el modelo puede ver.
2. El contexto es un presupuesto escaso — reserva espacio para la respuesta.
3. La fluidez no garantiza corrección.
